# Iterators, Generators & Lazy Pipelines (5+ Years Interview Guide)
Exhaustive revision guide to Iterator protocol (__iter__/__next__), generator functions (yield/yield from), generator pipelines, and .send() on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Iterator Protocol**: Dedicated cell for implementing `__iter__()` and `__next__()` with `StopIteration`.
- **Built-In Iterator Functions**: Dedicated cell for `iter()` and `next()`.
- **Generator Syntax**: Dedicated cell for `yield`, `yield from`, `.send()`, and `.close()`.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### The Iterator Protocol: `__iter__()` & `__next__()`
**Explanation**: An iterable implements `__iter__()` returning an iterator. An iterator implements `__next__()` yielding values sequentially and raising `StopIteration` when exhausted.

**Syntax**: `class CustomIterator: def __iter__(self): ... def __next__(self): ...`

In [2]:
class TransactionStream:
    def __init__(self, records, max_count=5):
        self.records = records
        self.max_count = min(max_count, len(records))
        self.idx = 0
    def __iter__(self):
        return self
    def __next__(self):
        if self.idx >= self.max_count:
            raise StopIteration
        item = self.records[self.idx]
        self.idx += 1
        return item['transaction_id'], float(item['transaction_amount'])

stream = TransactionStream(transactions, max_count=3)
for tx_id, amt in stream:
    print(f'Streamed via Protocol: {tx_id} -> ${amt:,.2f}')

Streamed via Protocol: TX110686 -> $1,216.33
Streamed via Protocol: TX107170 -> $324.99
Streamed via Protocol: TX108328 -> $136.66


### Built-in Iterator Functions: `iter()` and `next()`
**Explanation**: `iter(callable, sentinel)` creates sentinel-driven iterators. `next(it, default)` retrieves the next item with a fallback default on exhaustion.

**Syntax**: `next(iterator, default_val)`

In [3]:
tx_iter = iter(transactions[:3])
print('First item:', next(tx_iter)['transaction_id'])
print('Second item:', next(tx_iter)['transaction_id'])
print('Third item:', next(tx_iter)['transaction_id'])
print('Fourth item (exhausted with default):', next(tx_iter, 'NO_MORE_DATA'))

First item: TX110686
Second item: TX107170
Third item: TX108328
Fourth item (exhausted with default): NO_MORE_DATA


### Generator Functions (`yield`) & Sub-Generators (`yield from`)
**Explanation**: `yield` pauses execution and saves frame state. `yield from subgen` delegates iteration to a sub-generator seamlessly.

**Syntax**: `def gen(): yield x` / `yield from sub_iterable`

In [4]:
def batch_streamer(records, chunk_size=2):
    for i in range(0, 6, chunk_size):
        yield [r['transaction_id'] for r in records[i:i+chunk_size]]

def master_streamer(records):
    yield from batch_streamer(records, chunk_size=2)

for chunk in master_streamer(transactions):
    print('Yielded Batch Chunk:', chunk)

Yielded Batch Chunk: ['TX110686', 'TX107170']
Yielded Batch Chunk: ['TX108328', 'TX108563']
Yielded Batch Chunk: ['TX107002', 'TX113784']


### Coroutines & Two-Way Generators: `.send()` and `.close()`
**Explanation**: Generators accept values from callers via `val = yield expr` and `gen.send(data)`.

**Syntax**: `received = yield output` / `gen.send(input_val)`

In [5]:
def running_risk_tracker():
    total_spend = 0.0
    while True:
        amt = yield total_spend
        if amt is None:
            break
        total_spend += amt

tracker = running_risk_tracker()
next(tracker) # Prime the generator
print('After sending $100:', tracker.send(100.0))
print('After sending $250:', tracker.send(250.0))
tracker.close()

After sending $100: 100.0
After sending $250: 350.0


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Memory-Safe Streaming Multi-Gigabyte Log Pipelines
**Explanation**: Build a modular 3-stage lazy generator pipeline: (1) line reader -> (2) parser -> (3) fraud filter.

**Syntax**: `gen3 = (x for x in gen2 if condition)`

In [6]:
def stage1_read(records): yield from records
def stage2_parse(records): yield from ({'id': r['transaction_id'], 'amt': float(r['transaction_amount']), 'fraud': int(r['is_fraud'])} for r in records)
def stage3_filter(records): yield from (r for r in records if r['fraud'] == 1 and r['amt'] > 500.0)

pipeline = stage3_filter(stage2_parse(stage1_read(transactions)))
print('First High-Value Fraud via Pipeline:', next(pipeline))

First High-Value Fraud via Pipeline: {'id': 'TX110960', 'amt': 1805.16, 'fraud': 1}
